# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    # VS Code inyecta esta variable con la ruta absoluta del propio notebook,
    # así que la raíz del proyecto queda anclada a dónde vive el archivo .ipynb,
    # sin importar cuál sea el directorio de trabajo con el que arrancó el kernel
    # (que puede no ser la raíz del proyecto, según la configuración del editor).
    RAIZ_PROYECTO = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    RAIZ_PROYECTO = Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from modules.presentacion import (
    aplicar_tema_oscuro_notebook,
    ejecutar_extraccion_indice,
    exportar_csv_excel,
    exportar_ficha_a_pdf,
    exportar_xlsx,
    mostrar_emisoras,
    mostrar_ficha_completa_cliente,
    mostrar_ficha_rendimiento,
    probar_historial_dividendos,
    seleccionar_anio_interactivo,
    seleccionar_ticker_interactivo,
)
from modules.procesamiento import obtener_anios_disponibles

In [2]:
# Aplicar tema oscuro al notebook
aplicar_tema_oscuro_notebook()

# Configuración de rutas de salida
CARPETA_SALIDA = Path.cwd() / "output"
CARPETA_FICHAS_PDF = CARPETA_SALIDA / "fichas"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Ejecutar extracción
Consultar solo los lunes temprano para hacer un análisis rápido de las FIBRAS y para saber si hubo altas y bajas de emisoras.

In [ ]:
df = ejecutar_extraccion_indice(HEADLESS, TIMEOUT_DATOS_MS, CARPETA_SALIDA, EXPORTAR_CSV_ANALITICO)

### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [ ]:
if EXPORTAR_CSV_EXCEL:
    ruta_csv_excel = exportar_csv_excel(df, RUTA_CSV_EXCEL)
    print(f"CSV compatible con Excel guardado en: {ruta_csv_excel}")

if EXPORTAR_XLSX:
    ruta_xlsx = exportar_xlsx(df, RUTA_XLSX)
    print(f"Excel guardado en: {ruta_xlsx}")

## Consulta de emisoras

In [3]:
try:
    df
except NameError:
    df = None

df_emisoras = mostrar_emisoras(df, CARPETA_SALIDA)

Fuente de emisoras: histórico de 20260830_153020_list_of_tickers.csv (no se ejecutó la extracción de AMEFIBRA en esta corrida).
      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

### Ticker a consultar

Elige, del desplegable, el ticker a consultar (mismo listado de la sección "Consula de emisoras"). Al correr esta celda se despliega el selector; cambia la selección y luego corre la celda de abajo para consultar el ticker elegido.

In [4]:
selector_ticker = seleccionar_ticker_interactivo(df_emisoras["Emisora"])

Dropdown(description='Ticker:', options=('DANHOS13', 'EDUCA18', 'FIBRAMQ12', 'FIBRAPL14', 'FIBRAUP18', 'FIHO12…

In [5]:
TICKER_SELECCIONADO = selector_ticker.value
historial_dividendos = probar_historial_dividendos(TICKER_SELECCIONADO, df_emisoras["Emisora"], CARPETA_SALIDA)

Ticker probado: FNOVA17. Registros: 32
Periodicidad detectada: trimestral
CSV generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_160024_FNOVA17_dividendos.csv


,ticker,ex_date,amount_mxn,close_on_ex_date_mxn,yield_pct,annualized_yield_pct,periodicity
22,FNOVA17,2024-02-26,0.530664,28.75,1.845788,7.091711,trimestral
23,FNOVA17,2024-04-19,0.575027,28.58,2.011991,13.856164,trimestral
24,FNOVA17,2024-07-24,0.557374,28.15,1.980014,7.528179,trimestral
25,FNOVA17,2024-11-12,0.511633,28.15,1.817524,5.976543,trimestral
26,FNOVA17,2025-02-25,0.517047,28.15,1.836757,6.384916,trimestral
27,FNOVA17,2025-04-28,0.554076,28.15,1.968298,11.587563,trimestral
28,FNOVA17,2025-11-05,0.607154,28.15,2.156853,4.121734,trimestral
29,FNOVA17,2026-02-24,0.613360,28.15,2.178899,7.164847,trimestral
30,FNOVA17,2026-04-28,0.621634,28.15,2.208291,12.794069,trimestral
31,FNOVA17,2026-07-29,0.622228,28.15,2.210401,8.769527,trimestral


## FICHA DE DENDIMIENTO ANUAL PERSONALIZADO

La ficha usa el **año calendario** (`1 de enero` a `31 de diciembre`). Los pagos se filtran por `ex_date`, que es la fecha disponible en el historial de `yfinance`; no se inventa una fecha de pago que la fuente no proporciona. Los precios inicial y final son el primer y último cierre disponible dentro del año. El rendimiento por dividendos se calcula contra el precio inicial, y el rendimiento de capital contra la variación entre precio final e inicial. La ficha es informativa y no constituye una recomendación de inversión.

### Año a consultar

Elige, del desplegable, el año a consultar (solo se muestran los años con distribuciones disponibles para el ticker). Al correr esta celda se despliega el selector; cambia la selección y luego corre la celda de abajo para generar la ficha con el año elegido.

In [ ]:
AÑOS_DISPONIBLES = obtener_anios_disponibles(historial_dividendos)
selector_anio = seleccionar_anio_interactivo(AÑOS_DISPONIBLES)

In [ ]:
# Generamos la ficha de rendimiento anual y la exportamos a PDF
AÑO_SELECCIONADO = selector_anio.value
ruta_ficha = mostrar_ficha_rendimiento(TICKER_SELECCIONADO, AÑO_SELECCIONADO, CARPETA_SALIDA, historial_dividendos)

# Exportamos los resultados a un archivo pdf y mostramos la ruta del archivo generado
ruta_pdf_rendimiento = exportar_ficha_a_pdf(
    ruta_ficha, TICKER_SELECCIONADO, "rendimiento anual", AÑO_SELECCIONADO, CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_rendimiento}")

### Ficha completa del año seleccionado para cliente

Ficha completa anual pensada como entregable final para el cliente (escenario de inversión, distribuciones mensuales y rendimiento total en el año), con un diseño distinto al de la ficha de rendimiento anterior. Usa el mismo ticker y año ya elegidos arriba y los mismos datos reales (`historial_dividendos`); no inventa cifras. Es informativa y no constituye una recomendación de inversión.

In [ ]:
# Generamos la ficha completa anual para el cliente y la exportamos a PDF
ruta_ficha_completa_cliente = mostrar_ficha_completa_cliente(TICKER_SELECCIONADO, AÑO_SELECCIONADO, CARPETA_SALIDA, historial_dividendos)

# Exportamos los resultados a un archivo pdf y mostramos la ruta del archivo generado
ruta_pdf_completa_cliente = exportar_ficha_a_pdf(
    ruta_ficha_completa_cliente, TICKER_SELECCIONADO, "ficha completa cliente", AÑO_SELECCIONADO, CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_completa_cliente}")

## FICHA DE RENDIMIENTO Y RIESGO DE LOS ÚLTIMOS 12 MESES

Misma ficha de rendimiento de arriba, pero calculada sobre la ventana móvil de los últimos 12 meses completos (en vez de año calendario), con el riesgo mensual promedio del periodo (volatilidad del retorno total mensual: variación de precio + dividendos del mes) agregado como cifra destacada junto al rendimiento total.

In [6]:
# Fecha de referencia para la ventana móvil de 12 meses (fecha_fin del periodo).
# None = usa la fecha actual; fijar una fecha (ej. "2025-12-31") permite correr el
# análisis de forma retrospectiva, útil para pruebas. El flujo de año calendario
# de las celdas anteriores no se modifica y sigue disponible como antes.
from modules.procesamiento import calcular_ventana_movil_12_meses

FECHA_REFERENCIA_12M = None
FECHA_INICIO_12M, FECHA_FIN_12M = calcular_ventana_movil_12_meses(FECHA_REFERENCIA_12M)
print(f"Ventana de análisis: {FECHA_INICIO_12M:%Y-%m-%d} a {FECHA_FIN_12M:%Y-%m-%d}")

Ventana de análisis: 2025-08-31 a 2026-08-30


### Ficha sencilla de los últimos 12 meses

In [7]:
# Generamos la ficha sencilla de rendimiento de los últimos 12 meses y la exportamos a PDF
ruta_ficha_12m = mostrar_ficha_rendimiento(
    TICKER_SELECCIONADO, None, CARPETA_SALIDA, historial_dividendos, fecha_referencia=FECHA_REFERENCIA_12M
)

# Exportamos los resultados a un archivo pdf y mostramos la ruta del archivo generado
ruta_pdf_12m = exportar_ficha_a_pdf(
    ruta_ficha_12m, TICKER_SELECCIONADO, "rendimiento 12 meses", f"{FECHA_FIN_12M:%Y%m%d}", CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_12m}")

Ficha generada: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_160040_FNOVA17_20260830_ult12m_ficha_rendimiento.html


ex_date,amount_mxn,yield_pct
2025-11-05,$0.6072,2.16%
2026-02-24,$0.6134,2.18%
2026-04-28,$0.6216,2.21%
2026-07-29,$0.6222,2.21%


PDF generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-08-30_1600_FNOVA17_rendimiento-12-meses_20260830.pdf


### Ficha completa de los últimos 12 meses para cliente

Misma ficha completa de arriba, pero sobre la ventana móvil de últimos 12 meses: agrega una sección de riesgo del periodo con la volatilidad anualizada del retorno total mensual y la serie de los 12 retornos mensuales en barras (verde = mes positivo, rojo = mes negativo), colocada junto al desglose de rendimiento (plusvalía vs. distribuciones).

In [8]:
# Generamos la ficha completa de los últimos 12 meses para el cliente y la exportamos a PDF
ruta_ficha_completa_12m = mostrar_ficha_completa_cliente(
    TICKER_SELECCIONADO, None, CARPETA_SALIDA, historial_dividendos, fecha_referencia=FECHA_REFERENCIA_12M
)

# Exportamos los resultados a un archivo pdf y mostramos la ruta del archivo generado
ruta_pdf_completa_12m = exportar_ficha_a_pdf(
    ruta_ficha_completa_12m, TICKER_SELECCIONADO, "ficha completa 12 meses", f"{FECHA_FIN_12M:%Y%m%d}", CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_completa_12m}")

Ficha completa generada: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260830_160051_FNOVA17_20260830_ult12m_ficha_completa_cliente.html


PDF generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-08-30_1600_FNOVA17_ficha-completa-12-meses_20260830.pdf
